# Coverage-aware finite-sample risk allocation for SNIS
**Full follow-up experiment: controlled Gaussian mixtures + real UJIIndoorLoc Wi-Fi localization.**

This study retains the SNIS estimator and the fixed-budget optimization in the existing proposal. The candidate method is a structured risk surrogate, not a replacement for Monte Carlo and not a claimed novel theorem. We test whether posterior coverage information adds value beyond ordinary scale and generic risk fitting. The already-inspected UJIIndoorLoc official test set is a **reused benchmark**; new Monte Carlo draws do not make it an independent confirmation cohort. Synthetic final batches use new fixed seeds. Do not tune on either final set. A new external real-data cohort is required for confirmatory generalization claims.

For a fixed region $A$, write $\pi=p(A)$, $\rho=q(A)$, and $b=\pi^2\|\mu_A-\mu_{A^c}\|^2/d$. Under independent proposal draws, $P(\text{miss }A)=(1-\rho)^K$. The squared difference between the full posterior mean and the conditional posterior mean outside $A$ equals $b$. This is **not** the conditional SNIS squared error in general. For a fixed partition $\{A_j\}$, define $C_i(K)=\sum_j b_{ij}(1-\rho_{ij})^K$. Overlapping miss events can double-count, so this is a proxy, not a bound or an exact decomposition. Our candidate is
$$\widehat R_i(K)=a_i/K+\gamma\sum_j\widehat b_{ij}(1-\rho_{ij})^K.$$
Here $a_i,b_{ij},\gamma\ge0$, and $\rho_{ij}$ is computed from the fixed proposal, not from the target posterior at deployment. $\gamma$ is chosen on validation data only. The surrogate has decreasing marginal gains; the greedy budget allocator is exact for this surrogate. Arbitrary Monte Carlo risk curves use exact dynamic programming. Compare a matched-feature 1/K predictor, the original power-law predictor, a flexible direct-risk predictor, cheap rules, a privileged coverage surrogate, and an independently evaluated finite-sample oracle.

**Full configuration:** 4D and 8D, three regimes each, 12 fresh complete batches per synthetic case; 1,111 official Wi-Fi test records. Independent 256 + 256 oracle repetitions and 512 evaluation repetitions per observation.

**Run:** select GPU in Colab, then Runtime → Run all. No Drive required. The notebook downloads official UCI data; if automatic download fails, upload the official ZIP when prompted. It never silently substitutes synthetic data. Save the final ZIP. `FULL=True` is the shipped configuration. `FULL=False` is only a reduced debugging run on the same real data (not final evidence).

In [ ]:
# Install only missing dependencies (standard Colab usually has all of these).
import importlib.util, subprocess, sys
for module, package in [('numba','numba'),('sklearn','scikit-learn'),('scipy','scipy'),('joblib','joblib'),('pandas','pandas'),('matplotlib','matplotlib')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',package])

In [ ]:
import os,io,zipfile,urllib.request,json,time,hashlib,math,shutil,platform
os.environ.setdefault("OMP_NUM_THREADS","4")
os.environ.setdefault("MKL_NUM_THREADS","4")
from pathlib import Path
import numpy as np,pandas as pd,torch,matplotlib.pyplot as plt
from torch import nn
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from scipy.stats import t,spearmanr,binom
from scipy.special import ndtr,logsumexp
from numba import njit
from IPython.display import display
import joblib,itertools
FULL=True
SEED=20261123  # same fitted Wi-Fi pipeline as previous benchmark
NEW=20261207  # new sampling streams / synthetic cohorts
TEST_FIXTURE=False  # local implementation test only; never set for real study
KMIN,KMAX,KAVG=4,120,48; KS=np.arange(KMIN,KMAX+1); BATCH=32
NOPT=256 if FULL else 12; NEVAL=512 if FULL else 16; NTEACH=384 if FULL else 12; NVAL=256 if FULL else 12
EPOCHS=30 if FULL else 2
OUT=Path('/content' if Path('/content').exists() else '/tmp')/('coverage_aware_results_'+time.strftime('%Y%m%d_%H%M%S'));OUT.mkdir(exist_ok=False)
DEV=torch.device('cuda' if torch.cuda.is_available() else 'cpu');torch.manual_seed(SEED);np.random.seed(SEED)
if DEV.type=='cuda':torch.cuda.manual_seed_all(SEED)
else:torch.set_num_threads(4)
torch.backends.cuda.matmul.allow_tf32=False
start_total=time.perf_counter()
def T(x,dtype=torch.float32):return torch.as_tensor(x,dtype=dtype,device=DEV)
def sync():
    if DEV.type=='cuda':torch.cuda.synchronize()
print('Device:',DEV,'FULL:',FULL)


## 1. Mathematical and optimizer checks
A two-point target is useful because the exact risk is a binomial sum. Missing either state makes the estimator equal the other state. For $K=1$, the two coverage terms account for the entire risk; for larger $K$, outcomes in which both states are sampled can still have nonzero error.

In [ ]:
def two_point(K,pi,rho):
    j=np.arange(K+1); w1=pi/rho;w0=(1-pi)/(1-rho)
    est=j*w1/(j*w1+(K-j)*w0)
    exact=np.sum(binom.pmf(j,K,rho)*(est-pi)**2)
    coverage=pi*pi*(1-rho)**K+(1-pi)**2*rho**K
    return exact,coverage
for pi in [.1,.5,.9]:
    for rho in [.01,.2,.5,.99]:
        a,b=two_point(1,pi,rho);assert np.isclose(a,b)
        for k in [4,48,120]:
            a,b=two_point(k,pi,rho);assert a+1e-12>=b
for k in [4,48,120]:
    a,_=two_point(k,.3,.3);assert np.isclose(a,.3*.7/k)
print('Exact binomial identity checks passed.')


In [ ]:
@njit
def dp(risk,budget,kmin):
    n,L=risk.shape;extra=budget-n*kmin;mx=L-1;prev=np.full(extra+1,np.inf);prev[0]=0.;back=np.full((n,extra+1),-1,np.int32)
    for i in range(n):
        cur=np.full(extra+1,np.inf)
        for b in range(max(0,extra-(n-i-1)*mx),min(extra,(i+1)*mx)+1):
            for k in range(max(0,b-i*mx),min(mx,b)+1):
                z=prev[b-k]+risk[i,k]
                if z<cur[b]:cur[b]=z;back[i,b]=k
        prev=cur
    out=np.empty(n,np.int32);b=extra
    for i in range(n-1,-1,-1):k=back[i,b];out[i]=k+kmin;b-=k
    return out
import itertools
sm=np.array([[7.,4.,5.],[2.,6.,1.],[3.,1.,8.]]);k=dp(sm,5,1);assert sum(k)==5 and np.isclose(sum(sm[i,j-1] for i,j in enumerate(k)),min(sum(sm[i,j-1] for i,j in enumerate(z)) for z in itertools.product(range(1,4),repeat=3) if sum(z)==5))
def batches(n,seed):
    a=np.random.default_rng(seed).permutation(n);return [a[i:i+BATCH] for i in range(0,n,BATCH)]
@njit
def greedy(r,budget,kmin):
    n,L=r.shape;k=np.zeros(n,np.int32)
    for _ in range(budget-n*kmin):
        best=-1;gain=-np.inf
        for i in range(n):
            j=k[i]
            if j<L-1:
                z=r[i,j]-r[i,j+1]
                if z>gain:best=i;gain=z
        k[best]+=1
    return k+kmin
# Greedy is exact for diminishing marginal gains. Otherwise use unrestricted DP.
# This avoids charging the cheap baseline for an unnecessarily slow optimizer.
def alloc(r,bs):
    out=np.zeros(len(r),np.int32)
    for ix in bs:
        rr=np.ascontiguousarray(r[ix],dtype=np.float64);g=rr[:,:-1]-rr[:,1:]
        solver=greedy if np.all(g[:,:-1]>=g[:,1:]) else dp
        out[ix]=solver(rr,len(ix)*KAVG,KMIN)
    assert np.all((out>=KMIN)&(out<=KMAX)) and all(out[ix].sum()==len(ix)*KAVG for ix in bs);return out


## 2. Common coverage model and fair comparators
Each dataset supplies: cheap observation/proposal features $F$, a region feature array $H$ (one row per region), exact proposal masses $\rho$, and offline-only $b$ labels from reference posteriors. Region features never contain target posterior probabilities, means, likelihoods or risk. All learned comparators get the same observation/proposal information, including flattened region features and sorted region mass summaries; the structured model also uses the region-wise representation. Fit the coverage coefficient with supervised region labels; fit the residual amplitude using **out-of-fold** coverage predictions on teacher records, to reduce optimistic training errors. Select global $\gamma$ from the fixed nonzero grid on validation. A separate fallback can choose the cheap baseline via validation-selected shrinkage (alpha = 0). The unshrunk coverage policy remains the primary candidate even if it loses. A flexible direct-risk model is trained on a fixed coarse grid and interpolated; it is allowed nonmonotone risk curves. Privileged coverage fit is an offline diagnostic, using oracle-bank risk curves and true coverage coefficients; its allocations are evaluated on a separate bank. It cannot be deployed.

In [ ]:
from sklearn.model_selection import KFold
FLOOR=1e-10
GAMMAS=[.05,.1,.25,.5,1.,2.]
KNOTS=np.array([4,8,16,24,32,48,64,80,96,120]); KI=KNOTS-KMIN
ALPHAS=[0.,.25,.5,.75,1.]

def reg(seed):return HistGradientBoostingRegressor(max_iter=110 if FULL else 8,max_leaf_nodes=15,min_samples_leaf=15 if FULL else 5,l2_regularization=2,random_state=seed)
def pack(F,H,rho,b):
    F=np.asarray(F,float); H=np.asarray(H,float);rho=np.clip(np.asarray(rho,float),0,1);b=np.asarray(b,float)
    # region shape / mass summaries, all proposal-only
    sr=np.sort(rho,axis=1)[:,::-1];top=np.pad(sr[:,:min(5,sr.shape[1])],((0,0),(0,max(0,5-sr.shape[1]))))
    extra=np.column_stack([top,-np.sum(rho*np.log(np.maximum(rho,FLOOR)),1),np.sum(rho<.02,1),np.sum(rho**2,1)])
    global_f=np.column_stack([F,extra]); F=np.column_stack([global_f,H.reshape(len(H),-1)]);H=np.concatenate([H,np.broadcast_to(global_f[:,None,:],(*H.shape[:2],global_f.shape[1]))],axis=2)
    assert H.shape[:2]==rho.shape==b.shape and np.allclose(rho.sum(1),1,atol=1e-5) and np.isfinite(H).all() and np.isfinite(b).all()
    return dict(F=F,H=H,rho=rho,b=np.maximum(b,0))
def subset(x,ix):return {k:v[ix] for k,v in x.items()}
def cover(x,b):
    return sum(b[:,j,None]*np.power(1-x['rho'][:,j,None],KS[None,:]) for j in range(b.shape[1]))
def b_fit(x,seed):
    # Fit log(1+b/scale); bounded exponent prevents overflow on extrapolation.
    scale=max(float(np.median(x['b'][x['b']>0])) if np.any(x['b']>0) else 1.,FLOOR)
    h=x['H'].reshape(-1,x['H'].shape[-1]); y=np.log1p(x['b'].reshape(-1)/scale)
    return (reg(seed).fit(h,y),scale,float(y.max()))
def b_pred(m,x):
    r,s,cap=m; y=r.predict(x['H'].reshape(-1,x['H'].shape[-1]));return (s*np.expm1(np.clip(y,0,cap+1))).reshape(x['rho'].shape)
def b_oof(x,seed):
    out=np.empty_like(x['b']);kf=KFold(3,shuffle=True,random_state=seed)
    for tr,va in kf.split(x['F']):out[va]=b_pred(b_fit(subset(x,tr),seed),subset(x,va))
    return out
# fit the amplitude on relative-risk weighted squared error across K, with nonnegative constraint
# objective: sum_k [(a/K + gamma*C(K)-R(K))/max(R(K),floor)]^2

def fit_amplitude(R,C,gamma):
    w=1/np.maximum(R,np.maximum(np.median(R,1,keepdims=True)*.05,FLOOR))**2
    return np.maximum(np.sum(w*(R-gamma*C)/KS,axis=1)/np.sum(w/KS**2,axis=1),FLOOR)
def predict_amp(m,x):return np.exp(np.clip(m.predict(x['F']),-30,30))[:,None]/KS

def curve_models(x,R,seed):
    F=x['F']; logk=np.log(KS/KAVG); D=np.stack([np.ones_like(logk),-logk],1);Y=np.log(np.maximum(R,FLOOR));pa=Y@np.linalg.pinv(D).T;pa[:,1]=np.clip(pa[:,1],.05,2.5);pa[:,0]=np.mean(Y+pa[:,1,None]*logk,1)
    power=[reg(seed+i).fit(F,pa[:,i]) for i in range(2)]
    direct=[reg(seed+10+j).fit(F,Y[:,i]) for j,i in enumerate(KI)]
    inv=reg(seed+30).fit(F,np.log(fit_amplitude(R,np.zeros_like(R),0)))
    oof=b_oof(x,seed+50);C=cover(x,oof);bm=b_fit(x,seed+51)
    amps={g:reg(seed+60+j).fit(F,np.log(fit_amplitude(R,C,g))) for j,g in enumerate(GAMMAS)}
    return dict(power=power,direct=direct,inv=inv,b=bm,amps=amps)
def predict_curves(m,x):
    F=x['F'];logk=np.log(KS/KAVG); p=m['power'];z=np.exp(np.clip(p[0].predict(F)[:,None]-np.clip(p[1].predict(F),.05,2.5)[:,None]*logk,-30,30))
    lg=np.stack([r.predict(F) for r in m['direct']],1);flex=np.exp(np.stack([np.interp(np.log(KS),np.log(KNOTS),r) for r in lg]))
    C=cover(x,b_pred(m['b'],x));inv=predict_amp(m['inv'],x)
    out={'power_law':z,'direct_risk':flex,'inverse_K':inv}
    out.update({f'coverage_{g:g}':predict_amp(a,x)+g*C for g,a in m['amps'].items()});return out,C

def predict_one(m,x,k):
    F=x['F']
    if k=='inverse_K':return predict_amp(m['inv'],x)
    if k=='power_law':
        p=m['power'];return np.exp(np.clip(p[0].predict(F)[:,None]-np.clip(p[1].predict(F),.05,2.5)[:,None]*np.log(KS/KAVG),-30,30))
    if k=='direct_risk':
        lg=np.stack([r.predict(F) for r in m['direct']],1);return np.exp(np.stack([np.interp(np.log(KS),np.log(KNOTS),r) for r in lg]))
    g=float(k.split('_')[-1]);return predict_amp(m['amps'][g],x)+g*cover(x,b_pred(m['b'],x))

def value(r,k):return float(r[np.arange(len(k)),k-KMIN].mean())
def blend(a,b,alpha,bs):
    z=(1-alpha)*a+alpha*b;out=np.floor(z).astype(np.int32)
    for ix in bs:
        left=int(len(ix)*KAVG-out[ix].sum());order=ix[np.argsort(-(z[ix]-out[ix]),kind='stable')];out[order[:left]]+=1
    assert all(out[ix].sum()==len(ix)*KAVG for ix in bs) and np.all((out>=KMIN)&(out<=KMAX));return out

def select(m,x,R,cheap,bs):
    pred,_=predict_curves(m,x); policies={k:alloc(r,bs) for k,r in {**cheap,**pred}.items()};vals={k:value(R,v) for k,v in policies.items()}
    base=min(cheap,key=lambda k:vals[k]);cov=min((k for k in pred if k.startswith('coverage_')),key=lambda k:vals[k]);generic=min(['power_law','direct_risk','inverse_K'],key=lambda k:vals[k])
    alpha=min(ALPHAS,key=lambda a:value(R,blend(policies[base],policies[cov],a,bs)))
    return dict(base=base,cov=cov,generic=generic,alpha=alpha,val_risks=vals)

def policies_for(m,x,cheap,bs,sel,RA,RB,tau):
    pred,C=predict_curves(m,x);r={**cheap,**{k:pred[k] for k in ['power_law','direct_risk','inverse_K']},'coverage':pred[sel['cov']]};P={k:alloc(v,bs) for k,v in r.items()};P['coverage_shrunk']=blend(P[sel['base']],P['coverage'],sel['alpha'],bs)
    R=(RA+RB)/2; Ct=cover(x,x['b']);gamma=float(sel['cov'].split('_')[-1]); a=fit_amplitude(R,Ct,gamma);r['privileged_coverage']=a[:,None]/KS+gamma*Ct;r['privileged_inverse_K']=fit_amplitude(R,Ct,0)[:,None]/KS;r['asymptotic_oracle']=tau[:,None]/KS
    for k in ['privileged_coverage','privileged_inverse_K','asymptotic_oracle']:P[k]=alloc(r[k],bs)
    P['oracle_A']=alloc(RA,bs);P['oracle_B']=alloc(RB,bs);P['finite_oracle']=alloc(R,bs)
    return P,r,C,Ct

def corr(a,b):
    if np.std(a)==0 or np.std(b)==0:return None
    return float(spearmanr(a,b).statistic)
def diagnostics(pred,RE):
    rows=[];i=48-KMIN;j=24-KMIN;l=72-KMIN
    for k,r in pred.items():
        if k not in ['coverage','power_law','direct_risk','inverse_K','privileged_coverage','privileged_inverse_K','asymptotic_oracle']:continue
        rows.append(dict(method=k,median_relative_error=float(np.median(abs(r-RE)/np.maximum(RE,FLOOR))),risk48_spearman=corr(r[:,i],RE[:,i]),marginal24_72_spearman=corr(r[:,j]-r[:,l],RE[:,j]-RE[:,l])))
    return pd.DataFrame(rows)
def summarize(P,RE,bs,sel,errors=None):
    # For real data, paired repetitions conditional on fixed records; synthetic: complete independent batches.
    rows=[]; base=sel['base']; u=value(RE,P['uniform']); b=value(RE,P[base]); ob=value(RE,P['finite_oracle']);
    risk={k:RE[np.arange(len(v)),v-KMIN] for k,v in P.items()}
    for k,v in P.items():
        a=risk[k].mean(); batch=np.array([risk[k][ix].mean() for ix in bs]);bb=np.array([risk[base][ix].mean() for ix in bs]);harm=(batch-bb)/np.maximum(bb,FLOOR)
        if errors is not None: xx=errors[k].mean(1).astype(float);yy=errors[base].mean(1).astype(float)
        else:xx=batch;yy=bb
        xm=xx.mean();ym=yy.mean();gain=1-xm/ym;inf=-(xx-xm)/ym+xm*(yy-ym)/ym**2;h=t.ppf(.975,len(xx)-1)*inf.std(ddof=1)/np.sqrt(len(xx))
        rows.append(dict(method=k,mse=float(a),gain_vs_uniform_pct=100*(1-a/u),gain_vs_cheap_pct=100*(1-a/b),ci_low_pct=100*(gain-h),ci_high_pct=100*(gain+h),headroom_captured=(b-a)/(b-ob) if b>ob else np.nan,harmful_batches=float((harm>0).mean()),worst_decile_loss=float(np.sort(harm)[-max(1,math.ceil(.1*len(harm))):].mean()),at_min=int((v==KMIN).sum()),at_max=int((v==KMAX).sum())))
    return pd.DataFrame(rows).sort_values('mse')

def paired_coverage_vs_generic(P,RE,bs,sel,errors=None):
    k=sel['generic']; a=RE[np.arange(len(RE)),P['coverage']-KMIN];b=RE[np.arange(len(RE)),P[k]-KMIN]
    if errors is not None:x=errors['coverage'].mean(1);y=errors[k].mean(1)
    else:x=np.array([a[ix].mean() for ix in bs]);y=np.array([b[ix].mean() for ix in bs])
    x=x.astype(float);y=y.astype(float);xm=x.mean();ym=y.mean();g=1-xm/ym;z=-(x-xm)/ym+xm*(y-ym)/ym**2;h=t.ppf(.975,len(x)-1)*z.std(ddof=1)/np.sqrt(len(x));return dict(generic=k,gain_pct=100*g,ci_low_pct=100*(g-h),ci_high_pct=100*(g+h))
def save_case(name,P,R,bs,sel,curves,RE,C,Ct,errors=None):
    dest=OUT/name;dest.mkdir(exist_ok=True);s=summarize(P,RE,bs,sel,errors);s.to_csv(dest/'method_summary.csv',index=False);diagnostics(curves,RE).to_csv(dest/'risk_diagnostics.csv',index=False)
    pd.DataFrame(P).to_csv(dest/'allocations.csv',index=False);pd.DataFrame([{'method':k,'validation_mse':v} for k,v in sel['val_risks'].items()]).to_csv(dest/'validation_selection.csv',index=False)
    comp=paired_coverage_vs_generic(P,RE,bs,sel,errors);(dest/'selection.json').write_text(json.dumps({**sel,'coverage_vs_generic':comp},indent=2));np.savez_compressed(dest/'curves.npz',K=KS,reference=RE,predicted_coverage=curves['coverage'],predicted_power=curves['power_law'],predicted_coverage_term=C,privileged_coverage_term=Ct,oracle_A=R[0],oracle_B=R[1]);print(name,sel['base'],sel['cov'],sel['alpha'],comp);display(s.round(4));return s


## 3. Controlled Gaussian-mixture experiment
Original mixture and scale-matched separation/mismatch variants from the previous notebook. Define a fixed 5 × 3 partition of the first two coordinates around the nominal conditional center, with boundaries in units of the within-component posterior standard deviation (known from the model). This includes tail regions and sensitivity to perpendicular proposal mismatch. Their probabilities and truncated conditional means are analytically available offline. The allocator uses the proposal and known observation/context features, not target probabilities or posterior means. Fresh teacher, validation and 12 complete test batches of 84 per regime/dimension. The synthetic inference model is intentionally tractable; results measure allocation, not superiority to exact inference.

In [ ]:
MASTER=NEW
N=84
def rng_for(*keys):
    return np.random.default_rng(np.random.SeedSequence([MASTER,*map(int,keys)]))

def make_model(d,seed,regime,n=N):
    rng=np.random.default_rng(seed)
    c=rng.integers(0,3,n)
    modes=rng.choice(2,n,p=[.55,.45])
    latent_eps=rng.normal(size=(n,d)); obs_eps=rng.normal(size=(n,d))
    ctx=rng_for(11,d,seed)
    sep_idx=ctx.integers(0,3,n); mismatch_idx=ctx.integers(0,3,n)
    sigma=np.array([.45,1.,2.1])[c] if regime in ('original','noise_only') else np.ones(n)
    sep=np.array([.5,2.25,4.])[sep_idx] if regime in ('separation_only','combined_fixed_scale') else np.full(n,2.25)
    means=np.zeros((n,2,d)); means[:,0,0]=-sep; means[:,1,0]=sep
    x=latent_eps+means[np.arange(n),modes]+sigma[:,None]*obs_eps
    v=sigma**2/(1+sigma**2)
    a=np.array([1.,1.1,1.25])[c] if regime=='original' else np.full(n,1.1)
    u=v*a*a
    nominal=x/(1+sigma[:,None]**2)
    qmu=nominal*.88 if regime=='original' else nominal.copy()
    if regime in ('mismatch_only','combined_fixed_scale'):
        # deterministic direction perpendicular to mixture axis; offset is known to allocator
        qmu[:,1]+=np.array([0.,.5,1.5])[mismatch_idx]*np.sqrt(v)
    mismatch=np.linalg.norm(qmu-nominal,axis=1)/np.sqrt(v)
    # Exact conditional posterior; all likelihood normalizers common across components.
    lg=np.log([.55,.45])[None,:]-.5*np.sum((x[:,None,:]-means)**2,axis=2)/(1+sigma[:,None]**2)
    lg-=logsumexp(lg,axis=1,keepdims=True)
    gamma=np.exp(lg)
    pm=(x[:,None,:]+sigma[:,None,None]**2*means)/(1+sigma[:,None,None]**2)
    truth=np.sum(gamma[:,:,None]*pm,axis=1)
    post_scale=np.sqrt(d*v+np.sum(gamma*np.sum((pm-truth[:,None,:])**2,axis=2),axis=1))
    return dict(d=d,x=x,c=c,sigma=sigma,sep=sep,v=v,u=u,qmu=qmu,mismatch=mismatch,lg=lg,pm=pm,truth=truth,post_scale=post_scale)

def exact_tau2(m):
    # Closed form integral of p^2/q times squared distance to true posterior mean.
    v,u,d=m['v'],m['u'],m['d']; A=2/v-1/u
    assert np.all(A>0), 'Asymptotic variance does not exist for this proposal.'
    a=m['pm']-m['qmu'][:,None,:]; target=m['truth']-m['qmu']; terms=[]
    for j,k in itertools.product(range(2),repeat=2):
        h=(a[:,j]+a[:,k])/v[:,None]; center=h/A[:,None]
        c=(np.sum(a[:,j]**2+a[:,k]**2,axis=1)/v)-np.sum(h*h,axis=1)/A
        logc=-d*np.log(2*np.pi*v)+.5*d*np.log(2*np.pi*u)+.5*d*np.log(2*np.pi/A)-.5*c
        moment=d/A+np.sum((center-target)**2,axis=1)
        terms.append(m['lg'][:,j]+m['lg'][:,k]+logc+np.log(moment))
    t=np.exp(logsumexp(np.stack(terms,axis=1),axis=1))
    assert np.all(np.isfinite(t)&(t>0))
    return t



In [ ]:
def interval_moment(lo,hi,mean,sd):
    a=(lo-mean)/sd;b=(hi-mean)/sd
    mass=np.where(a>0,ndtr(-a)-ndtr(-b),ndtr(b)-ndtr(a));mass=np.maximum(mass,0)
    phi=lambda z:np.exp(-z*z/2)/np.sqrt(2*np.pi)
    return mass,mean*mass+sd*(phi(a)-phi(b))
def gaussian_pack(m):
    d=m['d'];x=m['x'];qmu=m['qmu'];u=m['u'];v=m['v'];pm=m['pm'];p=np.exp(m['lg']);mu=m['truth'];nom=x/(1+m['sigma'][:,None]**2)
    cuts=[[-np.inf,-2.,-1.,1.,2.,np.inf],[-np.inf,-1.5,1.5,np.inf]]
    masses=[];moments=[];rhos=[];hh=[]
    for j,k in itertools.product(range(5),range(3)):
        lows=[nom[:,0]+np.sqrt(v)*cuts[0][j],nom[:,1]+np.sqrt(v)*cuts[1][k]];highs=[nom[:,0]+np.sqrt(v)*cuts[0][j+1],nom[:,1]+np.sqrt(v)*cuts[1][k+1]]
        P=[];M=[];Q=[];QM=[]
        for axis in [0,1]:
            a,b=interval_moment(lows[axis][:,None],highs[axis][:,None],pm[:,:,axis],np.sqrt(v[:,None]));P.append(a);M.append(b)
            a,b=interval_moment(lows[axis],highs[axis],qmu[:,axis],np.sqrt(u));Q.append(a);QM.append(b)
        joint=P[0]*P[1];pi=np.sum(p*joint,1);moment=np.einsum('nj,nj,njd->nd',p,joint,pm);moment[:,0]=np.sum(p*M[0]*P[1],1);moment[:,1]=np.sum(p*M[1]*P[0],1)
        rho=Q[0]*Q[1];rhos.append(rho);masses.append(pi);moments.append(moment)
        hh.append(np.column_stack([rho,np.log(np.maximum(rho,1e-14)),QM[0]/np.maximum(Q[0],1e-14)-qmu[:,0],QM[1]/np.maximum(Q[1],1e-14)-qmu[:,1],np.full(len(x),j),np.full(len(x),k)]))
    pi=np.stack(masses,1);M=np.stack(moments,1);rho=np.stack(rhos,1);H=np.stack(hh,1);b=np.zeros_like(pi)
    assert np.allclose(pi.sum(1),1,atol=1e-10) and np.allclose(M.sum(1),mu,atol=1e-9)
    for j in range(pi.shape[1]):
        other=np.arange(pi.shape[1])!=j;outside=pi[:,other].sum(1);cm=M[:,other,:].sum(1)/np.maximum(outside[:,None],1e-300);b[:,j]=np.where(outside>1e-14,np.sum((mu-cm)**2,1)/d,0)
    F=np.column_stack([x,x*x,np.linalg.norm(x,axis=1),m['sigma'],np.sqrt(u),m['sep'],m['mismatch'],qmu])
    return pack(F,H,rho,b)

@torch.no_grad()
def gauss_bank(m,reps,seed):
    n=len(m['x']);d=m['d'];R=np.empty((n,len(KS)));gen=torch.Generator(device=DEV).manual_seed(seed)
    for st in range(0,n,8):
        sl=slice(st,min(st+8,n));size=len(m['x'][sl]);eps=torch.randn(size,reps,KMAX,d,device=DEV,generator=gen);u=T(m['u'][sl])[:,None,None,None];v=T(m['v'][sl])[:,None,None,None];z=T(m['qmu'][sl])[:,None,None,:]+u.sqrt()*eps
        pm=T(m['pm'][sl]);lg=T(m['lg'][sl]);lp=torch.logaddexp(lg[:,0,None,None]-.5*((z-pm[:,0,None,None,:])**2).sum(-1)/v[...,0],lg[:,1,None,None]-.5*((z-pm[:,1,None,None,:])**2).sum(-1)/v[...,0]);lw=lp+.5*d*(u[...,0]/v[...,0]).log()+.5*(eps**2).sum(-1)
        den=torch.logcumsumexp(lw,2);shift=z.amin(2,keepdim=True)-1.;est=(torch.logcumsumexp(lw[...,None]+(z-shift).log(),2)-den[...,None]).exp()+shift;err=((est-T(m['truth'][sl])[:,None,None,:])**2).mean(-1);R[sl]=err[:,:,KMIN-1:].mean(1).cpu().numpy()
    assert np.isfinite(R).all();return R

def cat_models(ms):return {k:np.concatenate([m[k] for m in ms],axis=0) if isinstance(ms[0][k],np.ndarray) else ms[0][k] for k in ms[0]}
def gauss_cheap(m):
    scores={'uniform':np.ones(len(m['x'])),'context_noise':m['sigma'],'proposal_width':np.sqrt(m['u']),'component_separation':m['sep']+.05,'proposal_mismatch':m['mismatch']+.05,'combined_context':np.sqrt(m['sigma']**2+m['sep']**2+m['mismatch']**2)}
    return {k:v[:,None]**2/KS for k,v in scores.items()}
SYN_ROWS=[]
for d in ([4,8] if FULL else [4]):
    for j,regime in enumerate(['original','separation_only','mismatch_only'] if FULL else ['original']):
        s=NEW+10000*d+1000*j;ntr=840 if FULL else 48;nva=336 if FULL else 24;nbatch=12 if FULL else 3;n=N if FULL else 12
        mt=make_model(d,s,regime,ntr);mv=make_model(d,s+1,regime,nva);mtst=cat_models([make_model(d,s+100+i,regime,n) for i in range(nbatch)])
        ft0,fv0,fx0=map(gaussian_pack,[mt,mv,mtst]);Rt0=gauss_bank(mt,NTEACH,s+2);Rv0=gauss_bank(mv,NVAL,s+3);m0=curve_models(ft0,Rt0,s+4);bv0=[np.arange(i,min(i+n,nva)) for i in range(0,nva,n)];sel0=select(m0,fv0,Rv0,gauss_cheap(mv),bv0)
        bx0=[np.arange(i*n,(i+1)*n) for i in range(nbatch)];RA0=gauss_bank(mtst,NOPT,s+6);RB0=gauss_bank(mtst,NOPT,s+7);RE0=gauss_bank(mtst,NEVAL,s+8)
        P0,r0,C0,Ct0=policies_for(m0,fx0,gauss_cheap(mtst),bx0,sel0,RA0,RB0,exact_tau2(mtst)/d)
        name=f'gaussian_{d}D_{regime}';res=save_case(name,P0,(RA0,RB0),bx0,sel0,r0,RE0,C0,Ct0);res['case']=name;SYN_ROWS.append(res);joblib.dump(m0,OUT/name/'learned_models.joblib')
syn=pd.concat(SYN_ROWS,ignore_index=True);syn.to_csv(OUT/'synthetic_summary.csv',index=False)


## 4. Real-data pipeline
Reuse the fitted-model specification from the earlier experiment. The official test records are never used for fitting, hyperparameter selection, or region construction in this notebook. The previous experiment already examined these records, so this analysis is exploratory follow-up. All teacher/validation/optimization/evaluation Monte Carlo banks use fresh streams.

In [ ]:
URLS=['https://archive.ics.uci.edu/static/public/310/ujiindoorloc.zip','https://archive.ics.uci.edu/ml/machine-learning-databases/00310/UJIIndoorLoc.zip']
def readzip(blob):
    d={}
    with zipfile.ZipFile(io.BytesIO(blob)) as z:
        for n in z.namelist():
            if n.lower().endswith('.zip'): d.update(readzip(z.read(n)))
            elif n.lower().endswith('trainingdata.csv'): d['train']=pd.read_csv(io.BytesIO(z.read(n)))
            elif n.lower().endswith('validationdata.csv'): d['test']=pd.read_csv(io.BytesIO(z.read(n)))
    if not d: raise ValueError('No official CSVs found')
    return d
cache=OUT/'uji.zip'; data=None; errors=[]
for url in ([None] if cache.exists() else [])+URLS:
    try:
        if url is None: blob=cache.read_bytes()
        else:
            with urllib.request.urlopen(urllib.request.Request(url,headers={'User-Agent':'Mozilla/5.0'}),timeout=90) as r: blob=r.read()
        data=readzip(blob); assert set(data)=={'train','test'}; cache.write_bytes(blob); break
    except Exception as e: data=None;errors.append(str(e))
if data is None:
    print('Download failed:',errors,'Upload the official ZIP from https://archive.ics.uci.edu/dataset/310/ujiindoorloc')
    from google.colab import files
    blob=next(iter(files.upload().values()));data=readzip(blob);cache.write_bytes(blob)
train,test=data['train'],data['test'];sha=hashlib.sha256(blob).hexdigest();assert (len(train),len(test))==(19937,1111)
for f in (train,test): f.columns=f.columns.str.strip().str.upper()
WAPS=sorted(c for c in train if c.startswith('WAP')); A=len(WAPS); LOC=['BUILDINGID','FLOOR','LONGITUDE','LATITUDE']
assert A==(12 if TEST_FIXTURE else 520); assert not train[WAPS+LOC].isna().any().any() and not test[WAPS+LOC].isna().any().any()
print('Training / official test:',len(train),len(test),'APs:',A,'SHA256:',sha)

In [ ]:
# Select disjoint user groups using original training data only. If infeasible, explicitly disclose row split.
def split(f):
    ids=np.arange(len(f));u=f.USERID.to_numpy();g=np.unique(u);s=(f.BUILDINGID.astype(str)+'_'+f.FLOOR.astype(str)).to_numpy();rng=np.random.default_rng(SEED);best=None;bs=1e9
    for _ in range(750):
        if len(g)<8:break
        q=rng.permutation(g);a=round(.6*len(g));b=max(1,round(.13*len(g))); parts=[ids[np.isin(u,h)] for h in [q[:a],q[a:a+b],q[a+b:a+2*b],q[a+2*b:]]]; sizes=np.array([len(x) for x in parts]);score=np.sum((sizes/len(f)-[.60,.13,.13,.14])**2)
        if min(sizes)>=(20 if TEST_FIXTURE else 250) and set(s[parts[0]])==set(s) and score<bs:best=parts;bs=score
    if best is not None:return best,'user-disjoint development'
    a,b=train_test_split(ids,test_size=.4,random_state=SEED,stratify=s);c,b=train_test_split(b,train_size=1/3,random_state=SEED+1,stratify=s[b]);d,e=train_test_split(b,test_size=.5,random_state=SEED+2,stratify=s[b]);return [a,c,d,e],'stratified rows; development users can overlap'
parts,split_type=split(train); fit,cal,teacher,val=[train.iloc[p].reset_index(drop=True) for p in parts]
def cap(f,n,seed):return f.sample(n=n,random_state=seed).reset_index(drop=True) if len(f)>n else f
cal=cap(cal,1600,SEED+3);teacher=cap(teacher,1000,SEED+4);val=cap(val,512,SEED+5)
assert len(np.unique(np.concatenate(parts)))==len(train)
print(split_type, [len(x) for x in [fit,cal,teacher,val,test]])
key=fit[LOC].to_numpy(np.float64);key[:,2:]=np.round(key[:,2:],2);nodes,labels=np.unique(key,axis=0,return_inverse=True);S=len(nodes);origin=nodes[:,2:].mean(0);XY=T(nodes[:,2:]-origin);groups,gi=np.unique(nodes[:,:2],axis=0,return_inverse=True);G=len(groups);GROUP=T(np.eye(G)[gi])
print('Survey locations:',S,'building-floor groups:',G)
def encode(f):
    raw=f[WAPS].to_numpy(np.float32);d=(raw!=100).astype(np.float32);r=np.where(d>0,(raw+75)/20,0);return d,r
def arrays(f):
    d,r=encode(f);gt=f[['LONGITUDE','LATITUDE']].to_numpy(np.float64)-origin;grp=np.full(len(f),-1)
    for j,g in enumerate(groups):grp[(f.BUILDINGID.to_numpy()==g[0])&(f.FLOOR.to_numpy()==g[1])]=j
    return dict(X=T(np.concatenate([d,r,r*r],1)),inp=T(np.concatenate([d,r],1)),gt=T(gt),group=grp,phone=f.PHONEID.to_numpy(),user=f.USERID.to_numpy())
fit_a,cal_a,teach_a,val_a,test_a=map(arrays,[fit,cal,teacher,val,test])

In [ ]:
# Independent Bernoulli detection and conditional Gaussian RSSI, smoothed toward floor and global statistics.
d,r=encode(fit)
def stats(index,size):
    n=np.bincount(index,minlength=size)[:,None];c=np.zeros((size,A));s=c.copy();ss=c.copy();np.add.at(c,index,d);np.add.at(s,index,r);np.add.at(ss,index,r*r);return n,c,s,ss
n,c,s,ss=stats(labels,S);ng,cg,sg,ssg=stats(gi[labels],G);C=c.sum(0);gp=(C+.5)/(len(d)+1);gm=s.sum(0)/(C+5);gm2=(ss.sum(0)+5)/(C+5)
pg=(cg+20*gp)/(ng+20);mg=(sg+10*gm)/(cg+10);mg2=(ssg+10*gm2)/(cg+10)
p=np.clip((c+8*pg[gi])/(n+8),1e-5,1-1e-5);mu=(s+8*mg[gi])/(c+8);v=np.clip((ss+8*mg2[gi])/(c+8)-mu*mu,.15**2,9.)
base=np.log1p(-p).sum(1);coef=np.concatenate([np.log(p)-np.log1p(-p)-.5*np.log(2*np.pi*v)-.5*mu*mu/v,mu/v,-.5/v],1);BASE=T(base);COEF=T(coef)
def raw(a):return a['X']@COEF.T+BASE
with torch.no_grad():
    logits=raw(cal_a);temps=[1.,2.,4.,8.,16.,32.,64.];scores=[float(((torch.softmax(logits/h,1)@XY-cal_a['gt'])**2).sum(1).mean()) for h in temps];TEMP=temps[int(np.argmin(scores))]
d0=cal_a['X'][0,:A].cpu().numpy();r0=cal_a['X'][0,A:2*A].cpu().numpy();check=(d0*np.log(p[0])+(1-d0)*np.log1p(-p[0])+d0*(-.5*np.log(2*np.pi*v[0])-.5*(r0-mu[0])**2/v[0])).sum();assert np.isclose(float(logits[0,0]),check,atol=.02,rtol=1e-5)
print('Calibrated likelihood temperature:',TEMP)

In [ ]:
model=nn.Sequential(nn.Linear(2*A,256),nn.LayerNorm(256),nn.ReLU(),nn.Dropout(.1),nn.Linear(256,128),nn.ReLU(),nn.Linear(128,S)).to(DEV);opt=torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=.0001);y=T(labels,torch.long)
for epoch in range(EPOCHS):
    model.train();loss_sum=0
    for ix in torch.randperm(len(y),device=DEV).split(256):
        loss=nn.functional.cross_entropy(model(fit_a['inp'][ix]),y[ix]);opt.zero_grad();loss.backward();opt.step();loss_sum+=float(loss.detach())*len(ix)
    if epoch==0 or (epoch+1)%5==0:print('Epoch',epoch+1,'CE',loss_sum/len(y))
model.eval()
with torch.no_grad():
    pc=torch.softmax(logits/TEMP,1);qc=model(cal_a['inp']);candidates=[(h,e) for h in [.75,1.,1.5,2.,3.,4.] for e in [.05,.15,.30]];sc=[float(-(pc*((1-e)*torch.softmax(qc/h,1)+e/S).log()).sum(1).mean()) for h,e in candidates];QT,EPS=candidates[int(np.argmin(sc))]
@torch.no_grad()
def proposal(a):return (1-EPS)*torch.softmax(model(a['inp'])/QT,1)+EPS/S
@torch.no_grad()
def prepare(a):
    lp=torch.log_softmax(raw(a)/TEMP,1);p=lp.exp();return dict(**a,lp=lp,p=p,q=proposal(a),ref=p@XY)
# Fit and calibration end here. Test posterior is used for offline evaluation only, never for deployable policy inputs.
teach_a,val_a,test_a=map(prepare,[teach_a,val_a,test_a]);assert torch.all(test_a['q']>0) and torch.allclose(test_a['p'].sum(1),torch.ones(len(test),device=DEV),atol=1e-5)
print('Proposal temperature and defensive mass:',QT,EPS)

In [ ]:
@torch.no_grad()
def features(a,q=None):
    q=proposal(a) if q is None else q;m=q@XY;var=(q@(XY*XY)-m*m).clamp_min(0).sum(1);ent=-(q*q.log()).sum(1);top=q.topk(2,1).values;d=a['inp'][:,:A];r=a['inp'][:,A:];count=d.sum(1);mean=r.sum(1)/count.clamp_min(1);sd=((r*r).sum(1)/count.clamp_min(1)-mean*mean).clamp_min(0).sqrt();bf=(q@GROUP).max(1).values
    return torch.stack([count,mean,sd,var.sqrt(),ent,top[:,0],top[:,0]-top[:,1],bf,(q*q).sum(1),m[:,0],m[:,1]],1).cpu().numpy()

def proxies(f):
    z={'uniform':np.ones(len(f)),'proposal_spread':f[:,3]+.1,'proposal_entropy':f[:,4]+.1,'inverse_AP_count':1/np.sqrt(f[:,0]+1),'proposal_ambiguity':1-f[:,5]+.05,'floor_uncertainty':1-f[:,7]+.05};return {k:s[:,None]**2/KS[None,:] for k,s in z.items()}

In [ ]:
# Prefix SNIS using log-cumulative sums; exact model posterior only for offline teacher/reference computation.
shift=XY.min(0).values-1;logxy=(XY-shift).log()
@torch.no_grad()
def bank(a,reps,seed,policies=None):
    gen=torch.Generator(device=DEV).manual_seed(seed);n=len(a['q']);R=np.empty((n,len(KS)),np.float64);names=list(policies or {});err={k:np.empty((reps,n),np.float32) for k in names};physical={k:np.empty((reps,n),np.float32) for k in names}
    for st in range(0,n,8):
        en=min(n,st+8);q=a['q'][st:en];m=len(q);cdf=q.cumsum(1);cdf[:,-1]=1;u=torch.rand((m,reps*KMAX),generator=gen,device=DEV);idx=torch.searchsorted(cdf.contiguous(),u.contiguous()).reshape(m,reps,KMAX).clamp_max(S-1);lw=(a['lp'][st:en]-q.log()).gather(1,idx.reshape(m,-1)).reshape(m,reps,KMAX);ld=torch.logcumsumexp(lw,2);est=torch.stack([(torch.logcumsumexp(lw+logxy[idx,j],2)-ld).exp()+shift[j] for j in range(2)],-1);e=((est-a['ref'][st:en,None,None,:])**2).sum(-1)/2;R[st:en]=e[:,:,KMIN-1:].mean(1).cpu().numpy()
        if names:
            pe=((est-a['gt'][st:en,None,None,:])**2).sum(-1).sqrt();rr=torch.arange(m,device=DEV)
            for k in names:
                kk=T(policies[k][st:en]-1,torch.long);err[k][:,st:en]=e[rr,:,kk].T.cpu().numpy();physical[k][:,st:en]=pe[rr,:,kk].T.cpu().numpy()
        if st==0 or en==n or st%400==0:print('Bank',seed, en,'/',n,flush=True)
    assert np.isfinite(R).all() and all(np.isfinite(z).all() for z in err.values());return R,err,physical
ix=T([0,1,2],torch.long);lw=T([-2.,-4.,-1.]);est=(torch.logsumexp(lw[:,None]+logxy[ix],0)-torch.logsumexp(lw,0)).exp()+shift;assert torch.allclose(est,torch.softmax(lw,0)@XY[ix],atol=.001,rtol=.0001)

### Fixed posterior regions and online features
Within each surveyed building-floor group, form at most three spatial clusters (K-means on **fit-map coordinates only**, never on test fingerprints). This yields a disjoint exhaustive partition. For each region compute $\rho_j=q(A_j)$ exactly from the neural proposal. Offline teacher labels use exact model posterior mass and conditional means. Region predictors use proposal mass, proposal conditional centroid, geometric region centroid and size, building-floor identity, and observation/proposal features. No target likelihood or posterior reference is accessed by deployable allocation policies.

In [ ]:
region=np.empty(S,int);nreg=0
for g in range(G):
    ix=np.flatnonzero(gi==g);nc=min(3,max(1,len(ix)//20));lab=KMeans(n_clusters=nc,random_state=SEED,n_init=10).fit_predict(nodes[ix,2:]) if nc>1 else np.zeros(len(ix),int)
    region[ix]=lab+nreg;nreg+=nc
REG=T(np.eye(nreg)[region]);RGEOM=(REG.T@XY)/REG.sum(0)[:,None];RSIZE=REG.sum(0)/S;RGROUP=(REG.T@GROUP).argmax(1)
assert torch.all(REG.sum(1)==1) and torch.all(REG.sum(0)>0)
@torch.no_grad()
def wifi_pack(a,q=None,offline=True):
    q=proposal(a) if q is None else q
    rho=q@REG; qm=torch.stack([(q*XY[:,j])@REG for j in range(2)],2)/rho.clamp_min(1e-12)[:,:,None];qbar=q@XY
    n=len(q);H=torch.cat([rho[:,:,None],rho.clamp_min(1e-12).log()[:,:,None],qm-qbar[:,None,:],RGEOM[None,:,:].expand(n,-1,-1),RSIZE[None,:,None].expand(n,-1,-1),nn.functional.one_hot(RGROUP,G).float()[None,:,:].expand(n,-1,-1)],2)
    b=torch.zeros_like(rho)
    if offline:
        p=a['lp'].double().softmax(1);xy=XY.double();rg=1-REG.double();outside=p@rg;M=torch.stack([(p*xy[:,j])@rg for j in range(2)],2);ref=p@xy;cm=M/outside.clamp_min(1e-300)[:,:,None];b=((ref[:,None,:]-cm)**2).sum(2)/2;b=torch.where(outside>1e-14,b,torch.zeros_like(b))
    return pack(features(a,q),H.cpu().numpy(),rho.cpu().numpy(),b.cpu().numpy())
print('Fixed regions:',nreg,'survey states:',S)
ft,fv,fx=[wifi_pack(a,a['q']) for a in [teach_a,val_a,test_a]]; Rt,_,_=bank(teach_a,NTEACH,NEW+100);Rv,_,_=bank(val_a,NVAL,NEW+101);bv=batches(len(fv['F']),SEED+200);m=curve_models(ft,Rt,NEW+50);sel=select(m,fv,Rv,proxies(fv['F']),bv)
# Ensure offline labels have no effect on deployable model outputs (online path constructs zero b placeholders).
check=wifi_pack({k:val_a[k][:8] for k in ['X','inp']},val_a['q'][:8],offline=False);assert np.allclose(check['F'],fv['F'][:8],atol=1e-5) and np.allclose(check['H'],fv['H'][:8],atol=1e-5)
for k,v in predict_curves(m,check)[0].items():assert np.allclose(v,predict_curves(m,subset(fv,np.arange(8)))[0][k])
print('Online features do not access target posterior. Frozen selection:',sel)
bx=batches(len(fx['F']),SEED+201);RA,_,_=bank(test_a,NOPT,NEW+102);RB,_,_=bank(test_a,NOPT,NEW+103)
with torch.no_grad():
    tau=((test_a['p']**2/test_a['q'])*((XY[None]-test_a['ref'][:,None])**2).sum(-1)/2).sum(1).cpu().numpy()
P,r,C,Ct=policies_for(m,fx,proxies(fx['F']),bx,sel,RA,RB,tau);RE,errs,locerr=bank(test_a,NEVAL,NEW+104,P)
wifi=save_case('wifi',P,(RA,RB),bx,sel,r,RE,C,Ct,errs);dest=OUT/'wifi';joblib.dump(m,dest/'learned_models.joblib')
pd.DataFrame([dict(method=k,mean_coordinate_error=float(v.mean()),median_coordinate_error=float(np.median(v))) for k,v in locerr.items()]+[dict(method='exact_posterior',mean_coordinate_error=float((test_a['ref']-test_a['gt']).norm(dim=1).mean()),median_coordinate_error=float((test_a['ref']-test_a['gt']).norm(dim=1).median()))]).to_csv(dest/'localization.csv',index=False)
np.savez_compressed(dest/'paired_mc_errors.npz',**errs);np.savez_compressed(dest/'teacher.npz',K=KS,risk=Rt,features=ft['F'],region_coefficients=ft['b'],proposal_region_mass=ft['rho']);np.savez_compressed(dest/'test_regions.npz',rho=fx['rho'],true_b=fx['b'],predicted_b=b_pred(m['b'],fx),region_id=region)

truth_b=fx['b'];estimated_b=b_pred(m['b'],fx);coverage_diag=dict(region_coefficient_spearman=corr(truth_b.ravel(),estimated_b.ravel()),coverage_at_48_spearman=corr(C[:,48-KMIN],Ct[:,48-KMIN]),coverage_marginal_24_72_spearman=corr(C[:,24-KMIN]-C[:,72-KMIN],Ct[:,24-KMIN]-Ct[:,72-KMIN]),oracle_A_B_mean_abs_count_difference=float(np.abs(P['oracle_A']-P['oracle_B']).mean()))
(dest/'coverage_diagnostics.json').write_text(json.dumps(coverage_diag,indent=2));print('Coverage diagnostics:',coverage_diag)


## 5. Runtime benchmark
Measure actual selected-state likelihood evaluations (no cached full posterior), including feature extraction and allocation. Exact enumeration is an essential competing direct inference method. Timing describes this hardware and implementation, not a general speed guarantee.

In [ ]:
@torch.no_grad()
def online_sample(a,q,k,seed):
    gen=torch.Generator(device=DEV).manual_seed(seed);n=len(k);mx=int(max(k));idx=torch.multinomial(q,mx,replacement=True,generator=gen);mask=torch.arange(mx,device=DEV)[None,:]<T(k,torch.long)[:,None];r,c=torch.where(mask);j=idx[r,c];lw=((a['X'][r]*COEF[j]).sum(1)+BASE[j])/TEMP-q[r,j].log();w=torch.full((n,mx),-torch.inf,device=DEV);w[r,c]=lw;return (torch.softmax(w,1)[:,:,None]*XY[idx]).sum(1)
ri=bx[0]; a={k:test_a[k][ri] for k in ['X','inp']};b0=[np.arange(len(ri))]
@torch.no_grad()
def online(k,seed):
    if k=='exact_posterior':return torch.softmax(raw(a)/TEMP,1)@XY
    q=proposal(a)
    if k=='uniform':counts=np.full(len(ri),KAVG,np.int32)
    elif k==sel['base']:counts=alloc(proxies(features(a,q))[k],b0)
    else:
        xx=wifi_pack(a,q,offline=False);counts=alloc(predict_one(m,xx,sel['cov'] if k in ['coverage','coverage_shrunk'] else k),b0)
        if k=='coverage_shrunk':counts=blend(alloc(proxies(features(a,q))[sel['base']],b0),counts,sel['alpha'],b0)
    return online_sample(a,q,counts,seed)
rt=[]
for k in dict.fromkeys(['uniform',sel['base'],'coverage','coverage_shrunk',sel['generic'],'exact_posterior']):
    online(k,NEW);sync()
    for j in range(12 if FULL else 2):
        sync();st=time.perf_counter();e=online(k,NEW+j+500);sync();assert torch.isfinite(e).all();rt.append(dict(method=k,repeat=j,milliseconds=1000*(time.perf_counter()-st)))
rt=pd.DataFrame(rt);rt.to_csv(dest/'runtime_trials.csv',index=False);runt=rt.groupby('method').milliseconds.agg(['median','min','max']).reset_index();runt.to_csv(dest/'runtime_summary.csv',index=False);display(runt)


## 6. Prespecified interpretation and export
The primary question is whether the unshrunk coverage policy beats the validation-selected cheap comparator **and** the validation-selected generic learned comparator. Report all settings, not only wins. The Wi-Fi interval is conditional Monte Carlo uncertainty, not population generalization; synthetic intervals use independent complete test batches. The numerical gate (2% gain over cheap, positive lower interval and positive lower interval versus generic) is exploratory and not multiplicity-adjusted. Capturing 25% of estimated oracle headroom is a secondary target. Privileged coverage success alone does not establish a usable method. A successful allocation does not imply faster end-to-end inference than exact summation. No novelty claim follows merely from this experiment.

In [ ]:
def saveplot(path):plt.tight_layout();plt.savefig(path,dpi=170);plt.show();plt.close()
for name,df in [('wifi',wifi)]+[(k,g) for k,g in syn.groupby('case')]:
    z=df.sort_values('gain_vs_cheap_pct');plt.figure(figsize=(9,5));x=z.gain_vs_cheap_pct.to_numpy();plt.barh(z.method,x);plt.axvline(0,color='black',lw=1);plt.axvline(2,color='gray',ls=':');plt.xlabel('MSE reduction vs selected cheap policy (%)');saveplot(OUT/name/'allocation_gain.png')
plt.figure(figsize=(8,5));plt.plot(KS,RE.mean(0),label='Independent MC reference');plt.plot(KS,r['coverage'].mean(0),label='Coverage');plt.plot(KS,r['power_law'].mean(0),label='Power law');plt.plot(KS,r['privileged_coverage'].mean(0),label='Privileged coverage');plt.yscale('log');plt.xlabel('K');plt.ylabel('Mean squared posterior estimation error');plt.legend();saveplot(dest/'risk_curves.png')
plt.figure(figsize=(6,5));plt.scatter(P['finite_oracle'],P['coverage'],s=9,alpha=.35);plt.plot([4,120],[4,120],'--');plt.xlabel('Estimated oracle count');plt.ylabel('Learned coverage count');saveplot(dest/'allocation_agreement.png')
plt.figure(figsize=(7,4));plt.barh(runt.method,runt['median']);plt.xlabel('Median online milliseconds / batch');saveplot(dest/'runtime.png')
rows=[]
for name,df in [('wifi',wifi)]+[(k,g) for k,g in syn.groupby('case')]:
    selection=json.loads((OUT/name/'selection.json').read_text());cv=df.set_index('method').loc['coverage'];cmp=selection['coverage_vs_generic'];rows.append(dict(case=name,cheap=selection['base'],generic=cmp['generic'],gamma=selection['cov'],alpha=selection['alpha'],gain_cheap_pct=cv.gain_vs_cheap_pct,ci_low_cheap=cv.ci_low_pct,gain_generic_pct=cmp['gain_pct'],ci_low_generic=cmp['ci_low_pct'],headroom_captured=cv.headroom_captured,exploratory_gate=bool(cv.gain_vs_cheap_pct>=2 and cv.ci_low_pct>0 and cmp['ci_low_pct']>0),capture25=bool(cv.headroom_captured>=.25)))
gate=pd.DataFrame(rows);gate.to_csv(OUT/'all_gates.csv',index=False);display(gate)
manifest=dict(full=FULL,real_data=not TEST_FIXTURE,dataset='UJIIndoorLoc',sha256=sha,train_rows=len(train),test_rows=len(test),regions=nreg,states=S,split=split_type,fit_rows=len(fit),teacher_rows=len(teacher),val_rows=len(val),seed=SEED,new_seed=NEW,device=str(DEV),torch_version=torch.__version__,nopt=NOPT,neval=NEVAL,nteach=NTEACH,nval=NVAL,epochs=EPOCHS,budget_avg=KAVG,kmin=KMIN,kmax=KMAX,real_test_reused=True,real_ci='conditional Monte Carlo',synthetic_ci='paired batch-level',elapsed_seconds=time.perf_counter()-start_total,gate='exploratory, unadjusted; no external confirmation')
(OUT/'manifest.json').write_text(json.dumps(manifest,indent=2));torch.save(model.cpu().state_dict(),dest/'proposal.pt');np.savez_compressed(dest/'target_model.npz',nodes=nodes,origin=origin,coef=coef,base=base,temperature=TEMP,proposal_temperature=QT,defensive_mass=EPS,region=region);np.savez_compressed(dest/'development_split_indices.npz',fit=parts[0],cal=parts[1],teacher=parts[2],validation=parts[3]);archive=OUT.parent/('Coverage_Aware_SNIS_Results.zip' if FULL and not TEST_FIXTURE else 'Coverage_Aware_DEBUG_Results.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for f in OUT.rglob('*'):
        if f.is_file() and f.suffix!='.zip':z.write(f,f.relative_to(OUT))
print('Saved:',archive)
if not TEST_FIXTURE:
    try:
        from google.colab import files
        files.download(str(archive))
    except ImportError:pass


In [ ]:

import json
from google.colab import files

uploaded = files.upload()

filename = "Final_Structured_Coverage_SNIS_Allocation.ipynb"
notebook = json.loads(uploaded[filename])

# Keep the final experiment's variables separate
# from the previous experiment.
namespace = {"__name__": "__main__"}

code_cells = [
    cell for cell in notebook["cells"]
    if cell["cell_type"] == "code"
]

for i, cell in enumerate(code_cells, 1):
    print(f"\nExecuting cell {i}/{len(code_cells)}")
    print("=" * 60)

    source = "".join(cell["source"])
    exec(compile(source, f"final_cell_{i}", "exec"), namespace)

print("Final experiment completed.")
